## Imports & Configuration

In [1]:
import pandas as pd
import numpy as np

from IPython.display import display

RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

# Source dataset (GitHub)
BASE_URL = (
    "https://raw.githubusercontent.com/"
    "semoglou/cart/main"
)

CSV_URL = (
    f"{BASE_URL}/llm_baselines/llm_taxonomies/"
    "CART_DeepSeek_R1_shared_paragraphs.csv"
)

# Expected systems
SYSTEMS = [
    "CART",
    "DeepSeek-R1"
]

# Expected dataset size
EXPECTED_ROWS = 3656

print("=" * 60)
print("EVALUATION DATA PREPARATION")
print("=" * 60)

print(f"Systems: {' vs '.join(SYSTEMS)}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Source: {CSV_URL}")

EVALUATION DATA PREPARATION
Systems: CART vs DeepSeek-R1
Random seed: 42
Source: https://raw.githubusercontent.com/semoglou/cart/main/llm_baselines/llm_taxonomies/CART_DeepSeek_R1_shared_paragraphs.csv


## Load & Validate Dataset

In [2]:
df = pd.read_csv(
    CSV_URL,
    dtype={"paragraph_id": "string"},
    encoding="utf-8-sig"
)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

required_columns = [
    "paragraph_id",
    "paragraph",
    "nf_cart",
    "nf_deepseek"
]

missing_columns = (
    set(required_columns) - set(df.columns)
)

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

df = df[required_columns].copy()

assert len(df) == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS:,} rows, "
    f"found {len(df):,}."
)

assert df.notna().all().all(), (
    "Dataset contains missing values."
)

assert df["paragraph_id"].is_unique, (
    "Dataset contains duplicate paragraph IDs."
)

for col in required_columns:

    assert (
        df[col]
        .astype("string")
        .str.strip()
        .ne("")
        .all()
    ), f"Empty values found in {col}."

# Check Uncategorized labels
label_columns = [
    "nf_cart",
    "nf_deepseek"
]

for col in label_columns:

    uncategorized = (
        df[col]
        .astype("string")
        .str.strip()
        .str.casefold()
        .eq("uncategorized")
    )

    assert not uncategorized.any(), (
        f"Uncategorized labels found in {col}."
    )

# Summary
print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)

print(f"Total paragraphs: {len(df):,}")

print(
    f"Unique paragraph IDs: "
    f"{df['paragraph_id'].nunique():,}"
)

print(
    f"Duplicate IDs: "
    f"{df['paragraph_id'].duplicated().sum()}"
)

print(
    f"Missing values: "
    f"{df.isna().sum().sum()}"
)

print("\nNarrative families per system:")

print(
    f"CART: "
    f"{df['nf_cart'].nunique()}"
)

print(
    f"DeepSeek-R1: "
    f"{df['nf_deepseek'].nunique()}"
)

print("\nDataset preview:")

display(df.head(10))

print("\nDataset validation passed.")

DATASET LOADED
Total rows: 3,656
Columns: ['paragraph_id', 'paragraph', 'nf_cart', 'nf_deepseek']

VALIDATION SUMMARY
Total paragraphs: 3,656
Unique paragraph IDs: 3,656
Duplicate IDs: 0
Missing values: 0

Narrative families per system:
CART: 10
DeepSeek-R1: 20

Dataset preview:


,paragraph_id,paragraph,nf_cart,nf_deepseek
0,2023-03-07_1_10_p3,"Νομίζω ότι είναι για όλους μας, για τη χώρα μα...",Railway safety remains neglected and unsecured,Political Responsibility for Systemic Failures
1,2023-03-07_1_10_p4,Προκύπτουν επομένως σοβαρά ερωτήματα για τις μ...,Systemic governance failures caused preventabl...,Systemic Governance Negligence Causation
2,2023-03-07_1_18_p2,"Κύριε Υπουργέ, κυρίες και κύριοι συνάδελφοι, κ...",Government obstructs crash investigation to ev...,Institutional Corruption and Impunity
3,2023-03-07_1_18_p4,"Κύριε Υπουργέ, συζητούμε ένα νομοσχέδιο σήμερα...",Negligence caused the preventable Tempi train ...,Independent Investigative Oversight Demands
4,2023-03-07_1_18_p7,Ας έρθουμε να δούμε τι έκανε η Κυβέρνηση τα τε...,Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation
5,2023-03-07_1_18_p8,Πριν από έξι με επτά μήνες περίπου είχαμε μάθε...,Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation
6,2023-03-07_1_18_p9,"Νέα παιδιά τα οποία παίρνουν το τρένο, φοιτητέ...",Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation
7,2023-03-07_1_29_p1,"Κύριε Πρόεδρε, αν και τα λόγια είναι περιττά σ...",Judicial and procedural obstacles to accountab...,Institutional Corruption and Impunity
8,2023-03-07_1_29_p3,"Ε.». Βεβαίως, αυτή η τραγωδία ήταν προδιαγεγρα...",Negligence caused the preventable Tempi train ...,Systemic Governance Negligence Causation
9,2023-03-07_1_31_p1,"Ευχαριστώ πολύ, κύριε Πρόεδρε. Κυρίες και κύρι...",Judicial and procedural obstacles to accountab...,Systemic Governance Negligence Causation



Dataset validation passed.


## Create Blind Annotation Dataset

In [3]:
# Blind Annotation Dataset
# CART vs DeepSeek-R1

annotation_rows = []
mapping_rows = []

for annotation_id, (_, row) in enumerate(
    df.iterrows(),
    start=1
):

    # Labels and source systems
    options = [
        ("CART", row["nf_cart"]),
        ("DeepSeek-R1", row["nf_deepseek"])
    ]

    # Randomize A/B order
    rng.shuffle(options)

    annotation_rows.append({

        "annotation_id": annotation_id,

        "paragraph": row["paragraph"],

        "option_A": options[0][1],

        "option_B": options[1][1],

        "option_C": "None of the above"

    })

    mapping_rows.append({

        "annotation_id": annotation_id,

        "paragraph_id": row["paragraph_id"],

        "system_A": options[0][0],

        "system_B": options[1][0]

    })

annotation_df = pd.DataFrame(annotation_rows)

mapping_df = pd.DataFrame(mapping_rows)

print("=" * 60)
print("BLIND ANNOTATION DATASET CREATED")
print("=" * 60)

print(
    f"Annotation rows: "
    f"{len(annotation_df):,}"
)

print(
    f"Mapping rows: "
    f"{len(mapping_df):,}"
)

print("\nBlind annotation dataset:")

display(annotation_df.head(10))

print("\nHidden system mapping:")

display(mapping_df.head(10))

BLIND ANNOTATION DATASET CREATED
Annotation rows: 3,656
Mapping rows: 3,656

Blind annotation dataset:


,annotation_id,paragraph,option_A,option_B,option_C
0,1,"Νομίζω ότι είναι για όλους μας, για τη χώρα μα...",Political Responsibility for Systemic Failures,Railway safety remains neglected and unsecured,None of the above
1,2,Προκύπτουν επομένως σοβαρά ερωτήματα για τις μ...,Systemic governance failures caused preventabl...,Systemic Governance Negligence Causation,None of the above
2,3,"Κύριε Υπουργέ, κυρίες και κύριοι συνάδελφοι, κ...",Government obstructs crash investigation to ev...,Institutional Corruption and Impunity,None of the above
3,4,"Κύριε Υπουργέ, συζητούμε ένα νομοσχέδιο σήμερα...",Negligence caused the preventable Tempi train ...,Independent Investigative Oversight Demands,None of the above
4,5,Ας έρθουμε να δούμε τι έκανε η Κυβέρνηση τα τε...,Systemic Governance Negligence Causation,Railway safety remains neglected and unsecured,None of the above
5,6,Πριν από έξι με επτά μήνες περίπου είχαμε μάθε...,Systemic Governance Negligence Causation,Railway safety remains neglected and unsecured,None of the above
6,7,"Νέα παιδιά τα οποία παίρνουν το τρένο, φοιτητέ...",Railway safety remains neglected and unsecured,Systemic Governance Negligence Causation,None of the above
7,8,"Κύριε Πρόεδρε, αν και τα λόγια είναι περιττά σ...",Institutional Corruption and Impunity,Judicial and procedural obstacles to accountab...,None of the above
8,9,"Ε.». Βεβαίως, αυτή η τραγωδία ήταν προδιαγεγρα...",Negligence caused the preventable Tempi train ...,Systemic Governance Negligence Causation,None of the above
9,10,"Ευχαριστώ πολύ, κύριε Πρόεδρε. Κυρίες και κύρι...",Judicial and procedural obstacles to accountab...,Systemic Governance Negligence Causation,None of the above



Hidden system mapping:


,annotation_id,paragraph_id,system_A,system_B
0,1,2023-03-07_1_10_p3,DeepSeek-R1,CART
1,2,2023-03-07_1_10_p4,CART,DeepSeek-R1
2,3,2023-03-07_1_18_p2,CART,DeepSeek-R1
3,4,2023-03-07_1_18_p4,CART,DeepSeek-R1
4,5,2023-03-07_1_18_p7,DeepSeek-R1,CART
5,6,2023-03-07_1_18_p8,DeepSeek-R1,CART
6,7,2023-03-07_1_18_p9,CART,DeepSeek-R1
7,8,2023-03-07_1_29_p1,DeepSeek-R1,CART
8,9,2023-03-07_1_29_p3,CART,DeepSeek-R1
9,10,2023-03-07_1_31_p1,CART,DeepSeek-R1


## Validate Annotation Dataset

In [4]:
assert len(annotation_df) == len(df)

assert len(mapping_df) == len(df)

assert len(annotation_df) == EXPECTED_ROWS


assert annotation_df["annotation_id"].is_unique

assert mapping_df["annotation_id"].is_unique


assert (
    annotation_df["annotation_id"].tolist()
    ==
    mapping_df["annotation_id"].tolist()
)


assert annotation_df.notna().all().all()

assert mapping_df.notna().all().all()


assert (
    annotation_df["option_C"]
    .eq("None of the above")
    .all()
)



expected_systems = {
    "CART",
    "DeepSeek-R1"
}

for _, row in mapping_df.iterrows():

    actual_systems = {
        row["system_A"],
        row["system_B"]
    }

    assert actual_systems == expected_systems



source_columns = {
    "CART": "nf_cart",
    "DeepSeek-R1": "nf_deepseek"
}

for i in range(len(df)):

    for option in ["A", "B"]:

        system = (
            mapping_df.iloc[i][f"system_{option}"]
        )

        actual_label = (
            annotation_df.iloc[i][f"option_{option}"]
        )

        expected_label = (
            df.iloc[i][source_columns[system]]
        )

        assert actual_label == expected_label, (
            f"Label mismatch at row {i}, "
            f"option {option}"
        )



print("=" * 60)
print("SYSTEM DISTRIBUTION ACROSS POSITIONS")
print("=" * 60)

for option in ["A", "B"]:

    print(f"\nPosition {option}:")

    print(
        mapping_df[
            f"system_{option}"
        ].value_counts()
    )



print("\n" + "=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

print(
    f"Validated paragraphs: "
    f"{len(annotation_df):,}"
)

print(
    f"Unique annotation IDs: "
    f"{annotation_df['annotation_id'].nunique():,}"
)

print("\nAll validation checks passed.")

SYSTEM DISTRIBUTION ACROSS POSITIONS

Position A:
system_A
DeepSeek-R1    1870
CART           1786
Name: count, dtype: int64

Position B:
system_B
CART           1870
DeepSeek-R1    1786
Name: count, dtype: int64

FINAL VALIDATION
Validated paragraphs: 3,656
Unique annotation IDs: 3,656

All validation checks passed.


## Export CSV Files

In [5]:
from pathlib import Path
from google.colab import files
import zipfile

OUTPUT_DIR = Path(
    "/content/evaluation_prepared"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INPUT_FILENAME = (
    "evaluation_input_cart_deepseek.csv"
)

MAPPING_FILENAME = (
    "evaluation_mapping_cart_deepseek.csv"
)

ZIP_FILENAME = (
    "evaluation_cart_deepseek_prepared.zip"
)

input_path = OUTPUT_DIR / INPUT_FILENAME

mapping_path = OUTPUT_DIR / MAPPING_FILENAME

zip_path = OUTPUT_DIR / ZIP_FILENAME

annotation_df.to_csv(
    input_path,
    index=False,
    encoding="utf-8-sig"
)

mapping_df.to_csv(
    mapping_path,
    index=False,
    encoding="utf-8-sig"
)

saved_input = pd.read_csv(input_path)

saved_mapping = pd.read_csv(mapping_path)

assert len(saved_input) == len(annotation_df)

assert len(saved_mapping) == len(mapping_df)

assert (
    saved_input.columns.tolist()
    ==
    annotation_df.columns.tolist()
)

assert (
    saved_mapping.columns.tolist()
    ==
    mapping_df.columns.tolist()
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    zipf.write(
        input_path,
        arcname=INPUT_FILENAME
    )

    zipf.write(
        mapping_path,
        arcname=MAPPING_FILENAME
    )

print("=" * 60)
print("EXPORT COMPLETED")
print("=" * 60)

print(f"\n1. {INPUT_FILENAME}")

print(
    f"   Rows: {len(saved_input):,}"
)

print(
    f"   Columns: "
    f"{saved_input.columns.tolist()}"
)

print(f"\n2. {MAPPING_FILENAME}")

print(
    f"   Rows: {len(saved_mapping):,}"
)

print(
    f"   Columns: "
    f"{saved_mapping.columns.tolist()}"
)

print("\nZIP archive:")

print(zip_path.name)

files.download(str(zip_path))

EXPORT COMPLETED

1. evaluation_input_cart_deepseek.csv
   Rows: 3,656
   Columns: ['annotation_id', 'paragraph', 'option_A', 'option_B', 'option_C']

2. evaluation_mapping_cart_deepseek.csv
   Rows: 3,656
   Columns: ['annotation_id', 'paragraph_id', 'system_A', 'system_B']

ZIP archive:
evaluation_cart_deepseek_prepared.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>